In [1]:
# =========================
# 1. IMPORTS
# =========================
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import mutual_info_classif

from xgboost import XGBClassifier


# =========================
# 2. LOAD DATA
# =========================
train_df = pd.read_csv('/kaggle/input/titanic/train.csv')

X = train_df.drop("Survived", axis=1)
y = train_df["Survived"]


# =========================
# 3. FEATURE ENGINEERING
# =========================

# Family features
X['FamilySize'] = X['SibSp'] + X['Parch'] + 1
X['IsAlone'] = (X['FamilySize'] == 1).astype(int)

# Encode Sex
X['Sex'] = X['Sex'].map({'male': 1, 'female': 0})

# Extract Title
X['Title'] = X['Name'].str.extract(r',\s*([^\.]+)\.')
X['Title'] = X['Title'].replace({
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
    'Lady': 'Rare', 'Countess': 'Rare', 'Capt': 'Rare',
    'Col': 'Rare', 'Don': 'Rare', 'Dr': 'Rare',
    'Major': 'Rare', 'Rev': 'Rare', 'Sir': 'Rare',
    'Jonkheer': 'Rare', 'Dona': 'Rare'
})

title_map = {'Mr':1, 'Miss':2, 'Mrs':3, 'Master':4, 'Rare':5}
X['Title'] = X['Title'].map(title_map)

# Cabin feature
X['CabinGroup'] = X['Cabin'].astype(str).str[0]
X['CabinGroup'] = X['CabinGroup'].replace('n', 'Unknown')

cabin_map = {'Unknown': 0, 'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7, 'T': 8}
X['CabinGroup'] = X['CabinGroup'].map(cabin_map)

# Interaction features
X['FarePerPerson'] = X['Fare'] / X['FamilySize']
X['Pclass_Fare'] = X['Pclass'] * X['FarePerPerson']

# Drop useless columns
X = X.drop(['Name', 'Ticket', 'Cabin'], axis=1)


# =========================
# 4. SPLIT DATA
# =========================
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# =========================
# 5. PREPROCESSING
# =========================
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])


# =========================
# 6. MODEL
# =========================
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.85,
        random_state=42,
        eval_metric='logloss'
    ))
])


# =========================
# 7. TRAIN + EVALUATE
# =========================
model.fit(X_train, y_train)

preds = model.predict(X_val)

accuracy = accuracy_score(y_val, preds)
print("Validation Accuracy:", accuracy)


# =========================
# 8. CROSS VALIDATION
# =========================
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=kf, scoring='accuracy')

print("CV Accuracy:", np.mean(cv_scores))


# =========================
# 9. FEATURE IMPORTANCE (MI)
# =========================
X_temp = X.copy()

for col in X_temp.select_dtypes(include=['number']).columns:
    X_temp[col] = X_temp[col].fillna(X_temp[col].median())

for col in X_temp.select_dtypes(include=['object']).columns:
    X_temp[col] = X_temp[col].fillna(X_temp[col].mode()[0]).astype('category').cat.codes

mi_scores = mutual_info_classif(X_temp, y)
mi_scores = pd.Series(mi_scores, index=X_temp.columns).sort_values(ascending=False)

print(mi_scores.head(10))

Validation Accuracy: 0.8268156424581006
CV Accuracy: 0.8170547988199109
Sex              0.167383
Title            0.156261
Fare             0.137366
FarePerPerson    0.129032
Pclass_Fare      0.114663
Pclass           0.042482
FamilySize       0.035032
Age              0.025904
IsAlone          0.023082
CabinGroup       0.022992
dtype: float64
